In [4]:
import pandas as pd
import json
from kafka import KafkaProducer
from time import time

# Load the parquet file
df = pd.read_parquet('dataset/green_tripdata_2025-10.parquet')

# Keep only the required columns
columns = [
    'lpep_pickup_datetime',
    'lpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'tip_amount',
    'total_amount',
]
df = df[columns]

# Convert datetime columns to strings
df['lpep_pickup_datetime'] = df['lpep_pickup_datetime'].astype(str)
df['lpep_dropoff_datetime'] = df['lpep_dropoff_datetime'].astype(str)

# Create Kafka producer
producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

# Create topic (if not already created)
# docker exec -it workshop-redpanda-1 rpk topic create green-trips

t0 = time()

for _, row in df.iterrows():
    producer.send('green-trips', value=row.to_dict())

producer.flush()

t1 = time()
print(f'took {(t1 - t0):.2f} seconds')
print(f'Total rows sent: {len(df)}')


took 10.41 seconds
Total rows sent: 49416
